In [ ]:
from brainspace.mesh.mesh_io import read_surface
from brainspace.utils.parcellation import map_to_labels
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.ticker import MultipleLocator
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
from surfplot import Plot

import cmasher as cmr
import glob
import matplotlib.pyplot as plt
import matplotlib as mpl
import natsort
import nibabel as nib
import numpy as np
import os
import pandas as pd
import pickle
import pingouin as pg
import seaborn as sns
import SUITPy.flatmap as flatmap
import warnings
warnings.filterwarnings('ignore')

def plot_subcortex(data, cmap, color_range, save_dir, surf_file='../utils/atlases/subcorticalSurf.pkl'):
    with open(surf_file, 'rb') as f:
        VL, VR = pickle.load(f)

    nROI = len(VL)

    cmap = cmap.copy()
    cmap.set_bad(color='grey')   
    cmap.set_under(color='grey')

    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(141, projection='3d') 
    ax2 = fig.add_subplot(142, projection='3d')  
    ax3 = fig.add_subplot(143, projection='3d') 
    ax4 = fig.add_subplot(144, projection='3d')  

    vmin, vmax = color_range
    def val_to_color(val):
        if val is None or np.isnan(val):
            return cmap(np.nan)
        frac = (val - vmin) / float(vmax - vmin)
        return cmap(frac)

    coords_ax1, coords_ax2, coords_ax3, coords_ax4 = [], [], [], []

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]    # left ROIs in second half
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax1.add_collection3d(mesh)
            coords_ax1.append(coords)

    ax1.view_init(elev=0, azim=0)
    ax1.set_box_aspect((1, 1, 1))
    ax1.set_axis_off()

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax2.add_collection3d(mesh)
            coords_ax2.append(coords)

    ax2.view_init(elev=0, azim=180)
    ax2.set_box_aspect((1, 1, 1))
    ax2.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]           # right ROIs in first half
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax3.add_collection3d(mesh)
            coords_ax3.append(coords)

    ax3.view_init(elev=0, azim=0)
    ax3.set_box_aspect((1, 1, 1))
    ax3.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax4.add_collection3d(mesh)
            coords_ax4.append(coords)

    ax4.view_init(elev=0, azim=180)
    ax4.set_box_aspect((1, 1, 1))
    ax4.set_axis_off()

    if coords_ax1:
        all_coords_1 = np.concatenate(coords_ax1, axis=0)
        ax1.auto_scale_xyz(all_coords_1[:, 0], all_coords_1[:, 1], all_coords_1[:, 2])

    if coords_ax2:
        all_coords_2 = np.concatenate(coords_ax2, axis=0)
        ax2.auto_scale_xyz(all_coords_2[:, 0], all_coords_2[:, 1], all_coords_2[:, 2])

    if coords_ax3:
        all_coords_3 = np.concatenate(coords_ax3, axis=0)
        ax3.auto_scale_xyz(all_coords_3[:, 0], all_coords_3[:, 1], all_coords_3[:, 2])

    if coords_ax4:
        all_coords_4 = np.concatenate(coords_ax4, axis=0)
        ax4.auto_scale_xyz(all_coords_4[:, 0], all_coords_4[:, 1], all_coords_4[:, 2])

    plt.tight_layout()
    plt.savefig(f"{save_dir}", dpi=800, transparent=True)
    plt.close()

def plot_cerebellum(df, value_col, color_range, cmap, save_path, label_file='../utils/atlases/atl-NettekovenAsym32_dseg.label.gii'):
    cerebellum_labels = ["region1", "region2", "region3", "region4", "region5", "region6", "region7", "region8", "region9", "region10", 
                         "region11", "region12", "region13", "region14", "region15", "region16", "region17", "region18", "region19", 
                         "region20", "region21", "region22", "region23", "region24", "region25", "region26", "region27", "region28", 
                         "region29", "region30", "region31", "region32"]

    vmin, vmax = color_range
    missing_val = vmin - 1.0
    roi_values = np.full(len(cerebellum_labels), np.nan, dtype=float)
    roi_map = dict(zip(df["roi"].astype(str), df[value_col].astype(float)))

    for i, label in enumerate(cerebellum_labels):
        if label in roi_map:
            roi_values[i] = roi_map[label]

    labels = nib.load(label_file)
    label_data = labels.darrays[0].data.astype(int)
    num_vertices = label_data.shape[0]
    data = np.full(num_vertices, missing_val, dtype=float)  

    for i, roi_value in enumerate(roi_values):
        mask = (label_data == (i + 1))
        if not np.any(mask):
            continue

        if np.isnan(roi_value):
            data[mask] = missing_val
        else:
            data[mask] = roi_value

    cmap = cmap.copy()
    cmap.set_under('grey')
    ax = flatmap.plot(data, overlay_type='func', cmap=cmap, cscale=(vmin, vmax), new_figure=True, colorbar=True, bordercolor='w')

    plt.savefig(save_path, dpi=800, transparent=True)
    plt.close()

def load_epoch_means(seed, e1, e2, fpc1_df):
    epoch_res = pd.read_csv("../output/subject_gradients.tsv", sep="\t")
    
    df = pd.read_csv(f"../output/seed/seed_{e1}_vs_{e2}_{seed}.tsv", sep="\t")
    df = df.merge(epoch_res[["roi_ix", "network"]], on="roi_ix", how="left")
    df = df.groupby(["network", "sub", "epoch", "task"], as_index=False)["r"].mean()
    df = (df.pivot_table(index=["sub", "epoch", "task"], columns="network", values="r").reset_index())
    df = (df.groupby(["sub", "epoch"], as_index=False).mean(numeric_only=True))
    df = df.merge(fpc1_df[["sub", "FPC1_z"]], on="sub", how="left")
    return df

def compute_within_subject_correlations(df, networks, e1, e2):
    wide = df.pivot(index="sub", columns="epoch", values=networks)
    delta = wide.xs(e2, level=1, axis=1) - wide.xs(e1, level=1, axis=1)
    delta.columns = [f"delta_{c}" for c in delta.columns]
    fpc = df.drop_duplicates("sub")[["sub", "FPC1_z"]].set_index("sub")
    return delta.join(fpc, how="left").reset_index()

def correlate_deltas(delta_df, networks):
    rows = []
    y = delta_df["FPC1_z"]

    for net in networks:
        x = delta_df[f"delta_{net}"]
        mask = x.notna() & y.notna()
        n = int(mask.sum())
        r, p = pearsonr(x[mask], y[mask])
        rows.append({"network": net, "r_delta_fpc1": r, "p_delta_fpc1": p})
    return pd.DataFrame(rows).sort_values("network")

def fdr_within_comparison(corr_df, p_col="p_delta_fpc1", alpha=0.05, method="fdr_bh"):
    pvals = corr_df[p_col].to_numpy()
    ok = np.isfinite(pvals)
    p_fdr = np.full_like(pvals, np.nan, dtype=float)
    reject = np.full(pvals.shape, False, dtype=bool)

    if ok.sum() > 0:
        reject_ok, p_fdr_ok, _, _ = multipletests(pvals[ok], alpha=alpha, method=method)
        p_fdr[ok] = p_fdr_ok
        reject[ok] = reject_ok

    corr_df = corr_df.copy()
    corr_df["p_fdr"] = p_fdr
    corr_df["reject_fdr"] = reject
    return corr_df
    
def yeo_cmap():
    cmap = {'cerebellum': (0, 0, 0), 'subcortex': (77, 77, 77), 'Vis': (119, 20, 140),
            'SomMot': (70, 126, 175), 'DorsAttn': (0, 117, 7), 'SalVentAttn': (195, 59, 255), 
            'Limbic': (219, 249, 165), 'Cont': (230, 149, 33), 'Default': (205, 65, 80)}
    cmap = {k: np.array(v) / 255 for k, v in cmap.items()}
    return cmap

def plot_corr_bars(corr_df, seed, comparison=None, cmap=None):
    dfp = corr_df.sort_values("r_delta_fpc1", ascending=False).copy()
    colors = [cmap.get(n, (0.5, 0.5, 0.5)) for n in dfp["network"]]

    fig, ax = plt.subplots(figsize=(3, 4))
    ax.bar(dfp["network"], dfp["r_delta_fpc1"], color=colors)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{seed} | {comparison}")
    ax.set_ylabel("corr(Δconnectivity, FPC1_z)")
    ax.set_xticklabels(dfp["network"], rotation=45, ha="right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    plt.tight_layout()
    plt.savefig(f"../figures/correlations-behaviour/{seed}_{comparison}.png", dpi=800, transparent=True)
    plt.close()

def plot_corr_polar(corr_df, seed, comparison=None, networks=None, color="grey", rlim=(-0.5, 0.5), tick_step=0.25):
    if networks is None:
        networks = ["subcortex", "cerebellum", "Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "SalVentAttn", "Default"]

    s = (corr_df.set_index("network")["r_delta_fpc1"].reindex(networks))

    values = s.to_numpy(dtype=float)
    values = np.nan_to_num(values, nan=0.0)

    angles = np.linspace(0, 2 * np.pi, len(networks), endpoint=False)
    angles_closed = np.r_[angles, angles[0]]
    values_closed = np.r_[values, values[0]]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
    ax.plot(angles_closed, values_closed, lw=2, color=color)
    eps = 1e-6  
    
    ax.fill_between(angles_closed, -eps, values_closed, color=color, alpha=0.20)
    ax.scatter(angles, values, s=45, facecolor="white", edgecolor=color, linewidth=1.2, zorder=3)

    ax.set_xticks(angles)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", pad=20)
    ax.set_ylim(*rlim)

    rticks = np.arange(rlim[0], rlim[1] + tick_step, tick_step)
    ax.set_yticks(rticks)
    ax.set_yticklabels([f"{t:.2f}" for t in rticks])

    for lab in ax.get_yticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontsize(25)

    ax.yaxis.grid(True, linestyle="--", linewidth=0.8, alpha=0.4)
    ax.xaxis.grid(False)
    
    ax.plot(angles_closed, np.zeros_like(angles_closed), color="black", lw=1.5, alpha=0.9, zorder=4)
        
    title = f"{seed} | {comparison}" if comparison else seed
    ax.set_title(title, pad=20)

    plt.tight_layout()
    plt.savefig(f"../figures/correlations-behaviour/{seed}_{comparison}_polar.png", dpi=800, transparent=True)
    plt.close(fig)

def _align_labels_to_atlas(x, source_labels, target_labels):
    target = np.unique(target_labels)[1:]
    df1 = pd.DataFrame(target, index=target)
    df2 = pd.DataFrame(x, index=source_labels)
    return pd.concat([df1, df2], axis=1).iloc[:, 1:].values
    
def weights_to_vertices(data, target, labels=None):
    if isinstance(target, str): 
        vertices = nib.load(target).get_fdata().ravel()
    else:
        vertices = target.ravel()
    if labels is not None:
        data = _align_labels_to_atlas(data, labels, vertices)
    mask = vertices != 0
    map_args = dict(target_lab=vertices, mask=mask, fill=np.nan)
    if (len(data.shape) == 1) or (data.shape[1] == 1):
        weights = map_to_labels(data.ravel(),  **map_args)
    else:
        weights = [map_to_labels(x, **map_args) for x in data.T]
    return weights


def get_surfaces(style='inflated', load=True):
    surf_path = '../utils/surfaces'
    surfaces = get_files([surf_path, f'*.{style}_*'])
    if load:
        surfs = [read_surface(i) for i in surfaces]
        return dict(zip(['lh', 'rh'], surfs))
    else:
        return surfaces
        
def get_files(pattern, force_list=False):
    if isinstance(pattern, list):
        pattern = os.path.join(*pattern)
    files = natsort.natsorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Pattern could not detect file(s): {pattern}")
    if (not force_list) and (len(files) == 1):
        return files[0]
    else:
        return files

def main():
    gradients = pd.read_csv("../output/subject_gradients.tsv", sep="\t")
    gradients = (gradients.loc[:, ["roi_ix", "roi", "network"]].drop_duplicates(subset="roi_ix"))

    networks = ["Cont","Default","DorsAttn","Limbic","SalVentAttn","SomMot","Vis","cerebellum","subcortex"]
    cmap = yeo_cmap()
    os.makedirs("../figures/correlations-behaviour", exist_ok=True)

    fpc1_df = pd.read_csv("../output/fpca_scores.csv")
    fpc1_df["sub"] = fpc1_df["subid"]
    fpc1_df = fpc1_df[["sub", "FPC1_z"]]

    all_corrs = []
    all_deltas = []
    for seed, col in zip(["7Networks_RH_Limbic_OFC_1", "7Networks_RH_SalVentAttn_FrOperIns_5" , "7Networks_LH_Default_Par_2", "7Networks_RH_DorsAttn_FEF_1"], ["#E1F8AD", "#B649F6", "#C1575D", "#337423"]):
        for e1, e2 in [("base", "early"), ("early", "late"), ("base", "late")]:
            target_networks = [n for n in networks if n != seed]
            df = load_epoch_means(seed, e1, e2, fpc1_df)

            delta_df = compute_within_subject_correlations(df, target_networks, e1, e2)
            delta_df["seed"] = seed
            delta_df["comparison"] = f"{e1}_vs_{e2}"
            all_deltas.append(delta_df)

            corr_df = correlate_deltas(delta_df, target_networks)
            corr_df = fdr_within_comparison(corr_df, p_col="p_delta_fpc1")
            corr_df["seed"] = seed
            corr_df["comparison"] = f"{e1}_vs_{e2}"
            all_corrs.append(corr_df)
            plot_corr_bars(corr_df, seed, comparison=f"{e1}_vs_{e2}", cmap=cmap)
            plot_corr_polar(corr_df, seed, comparison=f"{e1}_vs_{e2}", rlim=(-0.6, 0.6), tick_step=0.25)

            gradients_corr = gradients.merge(corr_df, on="network", how="left")
            gradients_corr = gradients_corr.rename(columns={"r_delta_fpc1": "r"})

            cortex = gradients_corr[gradients_corr["roi"].astype(str).str.startswith("7Net")].copy()
            cortex_labels = (cortex["roi_ix"].astype(int) - 64).to_list()
            
            vmax = 0.5
            brain_cmap = cmr.get_sub_cmap("bwr", 0.01, 0.99)
            surfaces = get_surfaces()

            seed_gradient = gradients[gradients["roi"] == f"{seed}"]
            seed_gradient["roi_ix"] = seed_gradient["roi_ix"] - 64
            seed_cmap = LinearSegmentedColormap.from_list("seed", [col, col], N=2)

            x = weights_to_vertices(cortex["r"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", cortex_labels)
            y = weights_to_vertices(np.array([1.0]), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", seed_gradient["roi_ix"].to_numpy())

            p = Plot(surfaces["lh"], surfaces["rh"])
            p.add_layer(x, cmap=brain_cmap, color_range=(-vmax, vmax), cbar=False)
            p.add_layer(y, cmap=seed_cmap, color_range=(0.5, 1.5), cbar=False)
            p.add_layer(y, cmap='binary_r', cbar=False, as_outline=True)

            _ = p.build(colorbar=False)
            plt.savefig(f"../figures/correlations-behaviour/{seed}_{e1}_vs_{e2}_brain.png", dpi=800, transparent=True)
            plt.close()

            r_subc = float(corr_df.loc[corr_df["network"] == "subcortex", "r_delta_fpc1"].iloc[0]) if "subcortex" in corr_df["network"].values else np.nan
            subc_data = np.full(64, r_subc, dtype=float)
            plot_subcortex(data=subc_data, cmap=brain_cmap, color_range=(-vmax, vmax), save_dir=f"../figures/correlations-behaviour/{seed}_{e1}_vs_{e2}_subcortex.png")
            
            r_cerb = float(corr_df.loc[corr_df["network"] == "cerebellum", "r_delta_fpc1"].iloc[0]) if "cerebellum" in corr_df["network"].values else np.nan
            cerb_plot_df = pd.DataFrame({"roi": [f"region{i}" for i in range(1, 33)], "r": [r_cerb] * 32})
            
            plot_cerebellum(df=cerb_plot_df, value_col="r", color_range=(-vmax, vmax), cmap=brain_cmap, save_path=f"../figures/correlations-behaviour/{seed}_{e1}_vs_{e2}_cerebellum.png")

    deltas_all = pd.concat(all_deltas, ignore_index=True)
    corrs_all = pd.concat(all_corrs, ignore_index=True)

    deltas_all.to_csv("../output/seed_network_delta_r.csv")
    corrs_all.to_csv("../output/seed_network_correlations_r.csv")

main()